## LLM 자동화: 원댓글 질문 판별 + 고정 카테고리 분류

- 입력: `text_clean`
- 출력: `is_question`, `question_category`, `question_confidence`, `llm_error`
- 질문이 아니면 `question_category = "해당없음"`
- 실제 전체 실행 전 아래 smoke test로 소량 검수 후 진행합니다.


In [34]:
# GEMINI API 설정
import json
import os
import time

from google import genai
from google.genai import types

from typing import Literal
from pydantic import BaseModel, Field

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError(
        "GEMINI_API_KEY 환경변수가 없습니다. API 키를 환경변수로 설정한 뒤 다시 실행해주세요."
    )

client = genai.Client()
MODEL_NAME = "gemini-3.1-flash-lite"

print("Gemini client 준비 완료")
print("사용 모델:", MODEL_NAME)

Gemini client 준비 완료
사용 모델: gemini-3.1-flash-lite


In [ ]:
QUESTION_CATEGORIES = (
    "직업_캐릭터",
    "육성_레벨링",
    "스펙업",
    "장비_아이템",
    "보스",
    "사냥",
    "스킬_6차",
    "재화_경제",
    "이벤트",
    "시스템",
    "기타",
    "해당없음",
)

QuestionCategory = Literal[
    "직업_캐릭터",
    "육성_레벨링",
    "스펙업",
    "장비_아이템",
    "보스",
    "사냥",
    "스킬_6차",
    "재화_경제",
    "이벤트",
    "시스템",
    "기타",
    "해당없음",
]


class QuestionClassification(BaseModel):
    is_question: bool

    question_category: QuestionCategory

    question_confidence: float = Field(ge=0.0, le=1.0)

In [ ]:
QUESTION_SYSTEM_PROMPT = """
너는 메이플스토리 YouTube 댓글을 분석하는 데이터 라벨러다.
각 댓글이 사용자가 정보나 조언을 얻기 위해 묻는 질문인지 판단하고,
질문이면 반드시 아래 고정 카테고리 중 하나만 선택한다.

카테고리 정의:
- 직업_캐릭터: 직업 선택, 캐릭터 추천, 직업 비교
- 육성_레벨링: 레벨업, 성장 순서, 뉴비 육성, 구간별 성장, 어떤 순서로 육성할지에 관한 질문, 특정 레벨 이후 무엇을 해야 하는지에 관한 질문
- 스펙업: 스탯, 방무, 보공, 환산, 스펙 상승 방향
- 장비_아이템: 장비를 어떻게 맞춰야 하는지에 대한 질문, 장비, 잠재, 옵션, 강화, 에테르넬, 칠흑 등
- 보스: 보스 공략, 보스 도전 스펙, 보스 관련 판단
- 사냥: 사냥터, 사냥 방식, 경험치, 사냥 효율, 메제
- 스킬_6차: 스킬, 코어, 5차/6차, 스킬 강화 순서, HEXA
- 메소마켓: 메소, 시세, 메이플포인트, 수수료, 메포
- 캐시샵: 코디템, 로얄스타일, 패스권, 펫, 마일리지, 캐시로 메포 교환하는 방법
- 경매장: 사고팔기에 적당한 가격인지에 대한 질문, 시세, 비용, 에르다조각, 아이템
- 이벤트 : 버닝, 이벤트, 서버, 게임 콘텐츠
- 시스템: 설정, 필터키, 게임 시스템, 이펙트, 소리
- 기타: 질문이지만 위 유형에 명확히 포함되지 않음
- 해당없음: 질문이 아님

카테고리 충돌 시 우선순위:
1. 시스템이 "왜 안 되는지", "어떻게 작동하는지"를 묻는 경우 -> 시스템
2. 캐릭터가 "앞으로 무엇을 해야 하는지", "어떻게 성장해야 하는지"를 묻는 경우 -> 육성_레벨링

판단 규칙:
1. 물음표가 없어도 질문 의도가 있으면 질문으로 판단한다.
2. 단순 감상, 감사, 주장, 답변, 정보 전달은 질문이 아니다.
3. 수사적 질문은 실제 정보 요청이 아니면 질문이 아니다.
4. is_question=false이면 question_category는 반드시 '해당없음'이다.
5. is_question=true이면 question_category는 '해당없음'이 될 수 없다.
6. question_confidence는 현재 판단에 대한 확신도를 0~1 사이 값으로 반환한다.
""".strip()


def validate_question_result(result):
    """LLM 결과가 프로젝트 규칙을 만족하는지 로컬에서 한 번 더 검증."""
    required = {
        "is_question",
        "question_category",
        "question_confidence",
    }
    if set(result) != required:
        raise ValueError(f"예상하지 못한 결과 필드: {set(result)}")

    if result["question_category"] not in QUESTION_CATEGORIES:
        raise ValueError(f"허용되지 않은 카테고리: {result['question_category']}")

    confidence = float(result["question_confidence"])

    if not 0 <= confidence <= 1:
        raise ValueError(f"confidence 범위 오류: {confidence}")

    result["question_confidence"] = confidence

    # 질문이 아니면 카테고리 강제 통일
    if result["is_question"] is False:
        result["question_category"] = "해당없음"
    # 질문인데 해당없음이면 비정상
    elif result["question_category"] == "해당없음":
        raise ValueError("질문인데 question_category가 '해당없음'으로 반환됨")

    return result


def classify_question_gemini(text_clean, max_retries=2):
    """댓글 1개를 한 번의 LLM 호출로 질문 여부와 카테고리까지 분류."""
    text_clean = str(text_clean).strip()

    if not text_clean:
        return {
            "is_question": False,
            "question_category": "해당없음",
            "question_confidence": 1.0,
        }

    prompt = f"""{QUESTION_SYSTEM_PROMPT}

                [분석할 댓글]
                {text_clean}"""

    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=QuestionClassification,
                    seed=42,
                ),
            )

            # Gemini SDK가 구조화된 응답
            # 자동 파싱한 경우
            if response.parsed is None:

                if isinstance(response.parsed, BaseModel):
                    result = QuestionClassification.model_validate_json(
                        response.parsed.model_dump()
                    )
                else:
                    result = response.parsed
            else:
                result = json.loads(response.text)

            print("Gemini raw text: ", response.text)
            print("Gemini parsed: ", response.parsed)
            return validate_question_result(result)

        except Exception as error:
            error_message = str(error)

            if "GenerateRequestsPerDayPerProjectPerModel-FreeTier" in error_message:
                print("일일 무료 요청 한도 소진")
                raise

            last_error = error

            print(f"오류 발생 ({attempt + 1}/{max_retries})")

            if attempt < max_retries - 1:
                time.sleep(5)

    raise RuntimeError(f"LLM 분류 실패: {text_clean[:80]}") from last_error

In [37]:
print(type(client))
print("models 있음:", hasattr(client, "models"))
print("responses 있음:", hasattr(client, "responses"))
print("모델:", MODEL_NAME)

<class 'google.genai.client.Client'>
models 있음: True
responses 있음: False
모델: gemini-3.1-flash-lite


In [47]:
test_text = "지금 막 시작했는데 메인 퀘 다 스킵하기가 안되던데 왜 그런건가요"

result = classify_question_gemini(test_text)

print(result)

오류 발생 (1)/2
ClientError
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite\nPlease retry in 8.845179032s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gem

RuntimeError: LLM 분류 실패: 지금 막 시작했는데 메인 퀘 다 스킵하기가 안되던데 왜 그런건가요

In [40]:
# 전체 실행 전에 5개만 먼저 확인하세요.
# 이 셀부터 실제 API 사용량이 발생합니다.
for sample_text in df_youtube_merged_sample["text_clean"].head(5):
    print("댓글:", sample_text)
    print("분류:", classify_question_gemini(sample_text))
    print("-" * 60)

댓글: 지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요
오류 발생 (1)/2
ClientError
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite\nPlease retry in 18.784075377s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimension

RuntimeError: LLM 분류 실패: 지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요

In [43]:
# LLM 결과 컬럼 초기화 + 체크포인트 재개
checkpoint_path = "../data/processed/youtube_question_llm_checkpoint.csv"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

if os.path.exists(checkpoint_path):
    print("기존 체크포인트를 불러옵니다.")
    df_youtube_merged_sample = pd.read_csv(checkpoint_path)

    # CSV 재로딩 시 True/False가 문자열이 되는 경우를 정리
    if "is_question" in df_youtube_merged_sample.columns:
        df_youtube_merged_sample["is_question"] = (
            df_youtube_merged_sample["is_question"]
            .map({True: True, False: False, "True": True, "False": False})
            .astype("boolean")
        )

    if "question_confidence" in df_youtube_merged_sample.columns:
        df_youtube_merged_sample["question_confidence"] = pd.to_numeric(
            df_youtube_merged_sample["question_confidence"],
            errors="coerce",
        )

if "is_question" not in df_youtube_merged_sample.columns:
    df_youtube_merged_sample["is_question"] = pd.Series(
        pd.NA, index=df_youtube_merged_sample.index, dtype="boolean"
    )

for column in ["question_category", "question_confidence", "llm_error"]:
    if column not in df_youtube_merged_sample.columns:
        df_youtube_merged_sample[column] = pd.NA

print("전체 행:", len(df_youtube_merged_sample))
print("이미 처리된 행:", df_youtube_merged_sample["is_question"].notna().sum())

기존 체크포인트를 불러옵니다.
전체 행: 2779
이미 처리된 행: 481


In [46]:
import time
import pandas as pd

MAX_REQUESTS_PER_RUN = 450
REQUEST_DELAY = 2

request_count = 0

for idx, row in df_youtube_merged_sample.iterrows():

    if pd.notna(row["is_question"]):
        continue

    if request_count >= MAX_REQUESTS_PER_RUN:
        print(f"오늘 설정한 최대 요청 수 " f"{MAX_REQUESTS_PER_RUN}건에 도달했습니다.")
        break

    try:
        result = classify_question_gemini(row["text_clean"])

        request_count += 1

        df_youtube_merged_sample.at[idx, "is_question"] = result["is_question"]

        df_youtube_merged_sample.at[idx, "question_category"] = result[
            "question_category"
        ]

        df_youtube_merged_sample.at[idx, "question_confidence"] = result[
            "question_confidence"
        ]

        df_youtube_merged_sample.at[idx, "llm_error"] = pd.NA

    except Exception as error:

        error_message = str(error)

        if "429" in error_message or "RESOURCE_EXHAUSTED" in error_message:
            print("Gemini quota 초과 → 저장 후 중단")
            break

        df_youtube_merged_sample.at[idx, "llm_error"] = error_message

    time.sleep(REQUEST_DELAY)


df_youtube_merged_sample.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

print("이번 실행 API 요청 수:", request_count)

print("현재까지 결과 저장:", checkpoint_path)

오류 발생 (1)/2
ClientError
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite\nPlease retry in 54.519066228s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'ge

KeyboardInterrupt: 

In [44]:
# 결과 검증 및 질문 댓글만 추출
invalid_false = df_youtube_merged_sample[
    (df_youtube_merged_sample["is_question"] == False)
    & (df_youtube_merged_sample["question_category"] != "해당없음")
]

invalid_true = df_youtube_merged_sample[
    (df_youtube_merged_sample["is_question"] == True)
    & (df_youtube_merged_sample["question_category"] == "해당없음")
]

assert invalid_false.empty, "질문이 아닌데 카테고리가 지정된 행이 있습니다."
assert invalid_true.empty, "질문인데 카테고리가 '해당없음'인 행이 있습니다."

print("LLM 오류 수:", df_youtube_merged_sample["llm_error"].notna().sum())
print("질문 수:", (df_youtube_merged_sample["is_question"] == True).sum())
print("질문 비율:", (df_youtube_merged_sample["is_question"] == True).mean())
print()

print("confidence 요약")
display(df_youtube_merged_sample["question_confidence"].describe())

print("카테고리 빈도")
display(
    df_youtube_merged_sample["question_category"]
    .value_counts(dropna=False)
    .rename_axis("question_category")
    .reset_index(name="count")
)

df_youtube_questions = df_youtube_merged_sample[
    df_youtube_merged_sample["is_question"] == True
].copy()

question_save_path = "../data/processed/youtube_questions_llm.csv"
df_youtube_questions.to_csv(
    question_save_path,
    index=False,
    encoding="utf-8-sig",
)

print("질문 데이터 저장:", question_save_path)

LLM 오류 수: 51
질문 수: 210
질문 비율: 0.4365904365904366

confidence 요약


count    481.000000
mean       0.977443
std        0.035436
min        0.800000
25%        0.950000
50%        1.000000
75%        1.000000
max        1.000000
Name: question_confidence, dtype: float64

카테고리 빈도


,question_category,count
0,NaN,2298
1,해당없음,271
2,육성_레벨링,39
3,보스,31
4,기타,26
5,장비_아이템,23
6,이벤트,22
7,사냥,20
8,시스템,19
9,스펙업,17


질문 데이터 저장: ../data/processed/youtube_questions_llm.csv


In [45]:
# 카테고리별 최대 10건씩 사람이 직접 샘플 검수
classified_rows = df_youtube_merged_sample.dropna(subset=["question_category"])

review_parts = []
for _, group in classified_rows.groupby("question_category"):
    review_parts.append(group.sample(min(len(group), 10), random_state=42))

if review_parts:
    review_sample = pd.concat(review_parts).sort_index()
else:
    review_sample = classified_rows.head(0).copy()

display(
    review_sample[
        [
            "text_raw",
            "text_clean",
            "is_question",
            "question_category",
            "question_confidence",
        ]
    ]
)

,text_raw,text_clean,is_question,question_category,question_confidence
0,지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요,지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요,True,시스템,1.00
6,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,True,스펙업,1.00
11,템화6.7이면 전투력이 2억이나 나와요?\r\n\r\n패파라서그런가 템환6.7로 1...,템화6 7이면 전투력이 2억이나 나와요? 패파라서그런가 템환6 7로 1 4억이던데,True,스펙업,0.95
15,선생님의 영상에 도움을 받아\r\n메린이가 마침내 챌린저 달성했어요\r\n그런데 이...,선생님의 영상에 도움을 받아 메린이가 마침내 챌린저 달성했어요 그런데 이제부터 뭘 ...,True,육성_레벨링,1.00
18,맑음님은 몇채널인지 공개하라 공개하라 !!!,맑음님은 몇채널인지 공개하라 공개하라,True,기타,0.80
...,...,...,...,...,...
475,2-3살자 빌드 이거 히트네 ㄷㄷ,2 3살자 빌드 이거 히트네 ㄷㄷ,False,해당없음,1.00
481,"2-6이 경치효율이 더 좋은데 왜 2-7 추천인거지,,,??",2 6이 경치효율이 더 좋은데 왜 2 7 추천인거지 ??,True,사냥,0.95
494,2-6아니였어??? 다들 2-6이라던데??,2 6아니였어?? 다들 2 6이라던데??,True,스킬_6차,0.95
500,"본인도 컴퓨터 계속 켜놓고 할거하면서 아이템 합성만 조지고 전사 궁수 40렙, 카오...",본인도 컴퓨터 계속 켜놓고 할거하면서 아이템 합성만 조지고 전사 궁수 40렙 카오스...,True,기타,0.95
